# Hackathon Judge Fine-tuning with Unsloth + SFT

Fine-tunes a small Qwen3 model to judge hackathon projects using:
- **Dataset**: `twangodev/madhacks25-synthetic-pairwise` — real MadHacks pairwise judgments from Qwen3.5-27B (the frontier model whose preferences we're distilling)
- **LoRA** for parameter-efficient fine-tuning
- **SFT** for supervised fine-tuning — trains the model to imitate the frontier model's full reasoning chain and verdict

**Runs on:** Free Google Colab T4 (16GB VRAM) or any GPU with 8GB+ VRAM.

## 1. Install Dependencies

In [ ]:
%%capture
!pip install unsloth unsloth_zoo trl datasets scikit-learn

## 2. Load the MadHacks Dataset

Each row is one pairwise judgment made by Qwen3.5-27B (the frontier model).
- `messages`: full conversation — system prompt + two project descriptions + frontier model's reasoning + verdict
- `verdict`: frontier model's choice (`A` or `B`) — this is our training label
- `position`: whether projects were shown as `ab` or `ba` (position-swapped pairs for bias checking)
- `gt_a_result` / `gt_b_result`: human award labels for each project

In [ ]:
from datasets import load_dataset
import pandas as pd

raw_ds = load_dataset("twangodev/madhacks25-synthetic-pairwise", split="train")
df = raw_ds.to_pandas()

print(f"Dataset shape: {df.shape}")
print(f"Columns: {df.columns.tolist()}")
print(f"Verdict distribution: {df['verdict'].value_counts().to_dict()}")
print(f"Unique pair_ids: {df['pair_id'].nunique()} (each pair appears twice with positions swapped)")
print(f"Frontier model: {df['model'].unique().tolist()}")

sample = df.iloc[0]
print(f"\nSample row:")
print(f"  pair_id:  {sample['pair_id']}")
print(f"  position: {sample['position']}")
print(f"  verdict:  {sample['verdict']}")
print(f"  gt_a:     {sample['gt_a_result']}")
print(f"  gt_b:     {sample['gt_b_result']}")

## 3. Format Dataset for SFT

SFT trains the model to reproduce the frontier model's full reasoning chain and verdict.
We keep the complete `messages` (system + user + assistant) — unlike GRPO which strips the assistant turn.

We split by `pair_id` so both position-swapped versions of a pair land in the same split.

In [ ]:
from datasets import Dataset
from sklearn.model_selection import train_test_split

def format_for_sft(row):
    """
    Keep the full messages including the assistant turn.
    SFT trains the model to reproduce the frontier model's reasoning + verdict.
    """
    return {
        "messages": row["messages"],   # system + user + assistant (full)
        "answer": row["verdict"],      # kept for eval only
        "pair_id": row["pair_id"],
        "position": row["position"],
        "gt_a_result": row["gt_a_result"],
        "gt_b_result": row["gt_b_result"],
    }

formatted = [format_for_sft(row) for _, row in df.iterrows()]

# Split by pair_id — keeps both position variants together
unique_pairs = df["pair_id"].unique()
train_pairs, test_pairs = train_test_split(unique_pairs, test_size=0.2, random_state=42)

train_data = [ex for ex in formatted if ex["pair_id"] in train_pairs]
test_data  = [ex for ex in formatted if ex["pair_id"] in test_pairs]

train_dataset = Dataset.from_list(train_data)
test_dataset  = Dataset.from_list(test_data)

print(f"Train: {len(train_dataset)} examples ({len(train_pairs)} unique pairs)")
print(f"Test:  {len(test_dataset)} examples ({len(test_pairs)} unique pairs)")
print(f"\nMessage roles: {[m['role'] for m in train_dataset[0]['messages']]}")
print(f"Frontier verdict (training target): {train_dataset[0]['answer']}")
print(f"\nAssistant response preview:")
print(train_dataset[0]['messages'][2]['content'][:300] + "...")

## 4. Load Model with Unsloth

In [ ]:
from unsloth import FastModel
import torch

# Qwen3-1.7B fits on a free Colab T4.
# Swap to unsloth/Qwen3-4B if you have more VRAM.
model, tokenizer = FastModel.from_pretrained(
    model_name = "unsloth/Qwen3-0.6B",  # smallest Qwen3 for testing
    # model_name="unsloth/Qwen3-1.7B",
    max_seq_length=4096,
    load_in_4bit=True,
    full_finetuning=False,
)

print(f"Model loaded on: {next(model.parameters()).device}")

## 5. Apply LoRA Adapters

In [ ]:
model = FastModel.get_peft_model(
    model,
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    lora_dropout=0.05,
    use_gradient_checkpointing="unsloth",
)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total     = sum(p.numel() for p in model.parameters())
print(f"Trainable: {trainable:,} / {total:,} ({100*trainable/total:.2f}%)")

## 6. Verdict Parser (for Evaluation)

We still need `parse_verdict` to evaluate the model's outputs at test time.
During SFT training this isn't used — the loss is cross-entropy on the assistant tokens only.

In [ ]:
import re

def parse_verdict(response: str):
    """
    Extract VERDICT: A/B/TIE from response.
    Matches the format used in the dataset's system prompt.
    """
    if "</think>" in response:
        response = response.split("</think>")[-1]

    match = re.search(r"VERDICT:\s*([AB]|TIE)", response, re.IGNORECASE)
    if match:
        return match.group(1).upper()

    # Fallback
    if re.search(r"\bProject\s+A\b", response, re.IGNORECASE): return "A"
    if re.search(r"\bProject\s+B\b", response, re.IGNORECASE): return "B"
    return None

# Quick check
print(parse_verdict("<think>reasoning</think>\nVERDICT: A"))  # A
print(parse_verdict("VERDICT: B"))                             # B
print(parse_verdict("I cannot decide"))                        # None

## 7. Train with SFT

`SFTTrainer` applies the chat template and trains only on the assistant tokens (not the prompt).
The model learns to reproduce the frontier model's reasoning and verdict.

In [ ]:
from trl import SFTTrainer, SFTConfig

def preprocess(example):
    text = tokenizer.apply_chat_template(
        example["messages"],
        tokenize=False,
        add_generation_prompt=False,
    )
    return {"text": text}

train_dataset_tokenized = train_dataset.map(preprocess)
test_dataset_tokenized  = test_dataset.map(preprocess)

training_args = SFTConfig(
    output_dir="./hackathon_judge_sft",
    num_train_epochs=3,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    warmup_steps=10,
    logging_steps=5,
    report_to="none",
    fp16=not torch.cuda.is_bf16_supported(),
    bf16=torch.cuda.is_bf16_supported(),
    max_seq_length=4096,
    dataset_text_field="text",  # point to the pre-formatted column
)

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    args=training_args,
    train_dataset=train_dataset_tokenized,
    # no formatting_func needed
)

print(f"Training on {len(train_dataset)} examples ({len(train_pairs)} unique pairs)")
print(f"Method: SFT — cross-entropy loss on assistant tokens only")
print(f"Epochs: {training_args.num_train_epochs}")
print("="*60)
trainer.train()

## 8. Save LoRA Weights

In [ ]:
# Get the underlying PEFT model and save it
trainer.model.save_pretrained("hackathon_judge_lora")
tokenizer.save_pretrained("hackathon_judge_lora")
print("Saved.")

## 9. Evaluate on Held-out Test Set

Measures agreement with the frontier model's verdict on unseen pairs.

In [ ]:
# Reload base model fresh, then attach the saved adapter
from unsloth import FastModel

model, tokenizer = FastModel.from_pretrained(
    model_name="unsloth/Qwen3-0.6B",
    max_seq_length=4096,
    load_in_4bit=True,
)

from peft import PeftModel
model = PeftModel.from_pretrained(model, "hackathon_judge_lora")
model.eval()
print("Loaded.")

def run_inference(prompt_messages):
    text = tokenizer.apply_chat_template(
        prompt_messages,
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=True,
    )
    inputs = tokenizer(text, return_tensors="pt").to(model.device)
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=1024,
            temperature=0.6,
            top_p=0.95,
            do_sample=True,
        )
    response = tokenizer.decode(outputs[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)
    return response, parse_verdict(response)


print("Test Set Evaluation")
print("="*60)

frontier_correct = 0
results = []

for ex in test_dataset:
    response, predicted = run_inference(ex["prompt"])
    frontier_match = predicted == ex["answer"]
    frontier_correct += int(frontier_match)

    think_text = ""
    if "<think>" in response and "</think>" in response:
        think_text = response.split("<think>")[1].split("</think>")[0].strip()

    results.append({
        "pair_id": ex["pair_id"],
        "position": ex["position"],
        "predicted": predicted,
        "frontier_verdict": ex["answer"],
        "gt_a": ex["gt_a_result"],
        "gt_b": ex["gt_b_result"],
        "frontier_match": frontier_match,
    })

    print(f"\npair {ex['pair_id'][:8]} | pos={ex['position']}")
    print(f"  Predicted: {predicted} | Frontier: {ex['answer']} | {'✓' if frontier_match else '✗'}")
    print(f"  GT A: {ex['gt_a_result']}")
    print(f"  GT B: {ex['gt_b_result']}")
    if think_text:
        print(f"  Reasoning: {think_text[:200]}...")

print("\n" + "="*60)
print(f"Agreement with frontier model: {frontier_correct}/{len(test_dataset)} = {frontier_correct/len(test_dataset)*100:.1f}%")

## 10. Position Bias Check

Each pair appears twice with projects in opposite order.
A consistent model should pick the same underlying project regardless of which slot it appears in.
Inconsistency = position bias (model favors whichever project appears first).

In [ ]:
results_df = pd.DataFrame(results)

print("Position Bias Analysis")
print("="*60)

n_consistent = 0
n_pairs_checked = 0

for pair_id in test_pairs:
    pair_rows = results_df[results_df["pair_id"] == pair_id]
    if len(pair_rows) < 2:
        continue

    ab_row = pair_rows[pair_rows["position"] == "ab"]
    ba_row = pair_rows[pair_rows["position"] == "ba"]
    if ab_row.empty or ba_row.empty:
        continue

    ab_verdict = ab_row.iloc[0]["predicted"]
    ba_verdict = ba_row.iloc[0]["predicted"]

    # Consistent: same underlying project wins in both orderings
    # ab=A means project_a_id won; ba=B means project_a_id won (since they were swapped)
    consistent = (ab_verdict == "A" and ba_verdict == "B") or \
                 (ab_verdict == "B" and ba_verdict == "A")

    n_consistent += int(consistent)
    n_pairs_checked += 1

    print(f"\npair {pair_id[:8]}:")
    print(f"  GT A: {ab_row.iloc[0]['gt_a']}")
    print(f"  GT B: {ab_row.iloc[0]['gt_b']}")
    print(f"  ab verdict: {ab_verdict} | ba verdict: {ba_verdict}")
    print(f"  Consistent: {'✓' if consistent else '✗ position bias detected'}")

print(f"\nConsistency: {n_consistent}/{n_pairs_checked} pairs = {n_consistent/n_pairs_checked*100:.0f}%")

## 11. Next Steps

**Scale the data**: Push your full DevPost scrape to HuggingFace in the same schema and swap the `load_dataset` call. Target 5k+ examples.

**Scale training**: Increase `num_train_epochs` to 5-10 with a larger dataset.

**Larger model**: Swap `Qwen3-0.6B` for `Qwen3-1.7B` or `Qwen3-4B` (needs ~10GB VRAM with 4-bit).

**Bradley-Terry ranking**: Convert all pairwise verdicts to a full ranking:

```python
# pip install choix
import choix, numpy as np

# comparisons: list of (winner_idx, loser_idx) tuples
params = choix.lsr_pairwise(n_items=num_projects, data=comparisons)
rankings = np.argsort(params)[::-1]
```

**Human alignment eval**: Compare BT rankings from each model against MadHacks human percentiles to measure which model best replicates human judgment.